# 单比特版图生成（标准化流程）
单比特版图生成标准化流程如下：
1. 设计约瑟夫森结并形成整合的gds文件以供芯片版图使用
2. 设计并初始化单比特版图并导入之前设计的约瑟夫森结
3. 编写标准化参数文件（excel）
4. 将参数文件转换为JSON格式的文件储存
5. 将JSON文件导入初始化的版图并形成N个分版图（比如将一整个晶圆分隔成N=9份，即3*3）
6. 加入磁通陷阱
7. 加入标志并形成总版图


In [1]:
# automatic reloading of modules when they change.
%load_ext autoreload
%autoreload 2

In [3]:
import qiskit_metal as metal
from qiskit_metal.qlibrary.user_components.reader import *
from qiskit_metal.qlibrary.user_components.writer import *
from qiskit_metal.qlibrary.user_components.layout import single_qubit_layout,build_airbridge
from qiskit_metal.qlibrary.user_components.my_qcomponent import JJTaper,JJManhattan,JJDolan
from qiskit_metal.qlibrary.user_components.save_load import *
from qiskit_metal.qlibrary.user_components.cell import *
from qiskit_metal import designs
import copy
import json
from qiskit_metal import draw, Dict,designs,MetalGUI
import gdspy

## 1. 参数化设计约瑟夫森结
这里的约瑟夫森结有三种，第一种是十字结，这个是最普遍的结，缺点是镀膜的时候容易局部断裂；第二种是多兰结，结的两个臂之间相差一定距离；
第三种是漏斗形结，也是做成多兰结的形式，不过尾部延伸为漏斗型结构而不是直线型结构以降低二能级损耗

In [3]:
design = metal.designs.DesignPlanar()
gui = metal.MetalGUI(design)
design.overwrite_enabled = True

  return original_import(name, *args, **kwargs)



### 1.1 结的种类(结文件分Cell设计)
#### 1.1.1 十字结
十字结是漏洞型结的对照组，由于要保持上下比特结方向一致，需要做正向结及反向结两种结构

In [ ]:
jj2 = JJManhattan(design, 'jj_manhattan',options=Dict(pos_x='0um',pos_y='0um',JJ_pad_up_height='30um',JJ_pad_lower_height='25um',orientation='-90'))
gui.rebuild()
gui.autoscale()
gui.zoom_on_components(['jj_manhattan'])

jj_path = ''
a_gds = writer_planar(design,jj_path)
a_gds.export_to_gds('gds/temp/JJ_Manhattan.gds')

design.components['jj_manhattan'].options['orientation']='90'
gui.rebuild()
a_gds.export_to_gds('gds/temp/JJ_Manhattan_flip.gds')

#### 1.1.2 多兰结
多兰结是漏洞型结的对照组，如果在同一版图中漏斗型结为多兰桥结构，那对照组也应做成多兰桥结构以保证工艺的一致性
多兰结也有正向和反向两种结构

In [4]:
jj2 = JJDolan(design, 'jj_dolan',options=Dict(pos_x='0um',pos_y='0um',JJ_pad_up_height='30um',JJ_pad_up_ext_width='22.15um',JJ_pad_up_ext_height='7um',JJ_pad_lower_height='25um',
                                              finger_lower_width='0.18um',finger_up_width='0.18um',extension_up='2um',extension_lower='-0.24um',orientation='-90'))
gui.rebuild()
gui.autoscale()
gui.zoom_on_components(['jj_dolan'])

jj_path = ''
a_gds = writer_planar(design,jj_path)
a_gds.export_to_gds('gds/temp/JJ_Dolan.gds')

design.components['jj_dolan'].options['orientation']='90'
gui.rebuild()
a_gds.export_to_gds('gds/temp/JJ_Dolan_flip.gds')

09:52AM 16s WARNING [_qgeometry_to_gds]: Unexpected shapely object geometry.The variable qgeometry_element is <class 'numpy.float64'>, method can currently handle Polygon and FlexPath.
09:52AM 16s WARNING [_import_junction_gds_file]: Not able to find file:"".  Not used to replace junction. Checked directory:"D:\Users\ls\PycharmProjects\pythonProject".
09:52AM 16s WARNING [_qgeometry_to_gds]: Unexpected shapely object geometry.The variable qgeometry_element is <class 'numpy.float64'>, method can currently handle Polygon and FlexPath.
09:52AM 16s WARNING [_import_junction_gds_file]: Not able to find file:"".  Not used to replace junction. Checked directory:"D:\Users\ls\PycharmProjects\pythonProject".


1

#### 1.1.3 漏斗结
单比特版图主要测试漏斗结的效果是否可以减少损耗，增加T1时间以及比特的存活率
同时，需要测试不同结线宽对于电感（比特频率）的影响，因此需要设计不同的结线宽


In [5]:
# design.delete_all_components()
# # 生成5个不同的结线宽gds版图
# JJ_up_linewidth = JJ_lower_linewidth= str(0.16)+'um'
# jj_path = ''
# for i in range(5):
#     jj = JJTaper(design, 'jj_taper', options=Dict(JJ_space_x='0.15um',JJ_space_y='0.8um',fillet='0.05um',
#                                                    JJ_up_linewidth=JJ_up_linewidth,JJ_lower_linewidth=JJ_lower_linewidth,
#                                                    orientation_angle='0',chip='main'))
#     a_gds = writer_planar(design,jj_path)
#     a_gds.export_to_gds('gds/temp/JJ_taper_'+str(round(0.16+i*0.01,2))+'_test.gds')
#     JJ_up_linewidth =JJ_lower_linewidth=str(round(0.16+(i+1)*0.01,2))+'um'
# 
# #其中4号和5号比特由于是反向的设置为反向结，选择180度
# design.components['jj_taper'].options.orientation='180'
# design.components['jj_taper'].options.JJ_up_linewidth='0.18um'
# design.components['jj_taper'].options.JJ_lower_linewidth='0.18um'
# gui.rebuild()
# a_gds.export_to_gds('gds/temp/JJ_taper_'+str(round(0.18,2))+'r_test.gds')
# 
# design.components['jj_taper'].options.orientation='180'
# design.components['jj_taper'].options.JJ_up_linewidth='0.19um'
# design.components['jj_taper'].options.JJ_lower_linewidth='0.19um'
# gui.rebuild()
# a_gds.export_to_gds('gds/temp/JJ_taper_'+str(round(0.19,2))+'r_test.gds')
# 
# design.components['jj_taper'].options.orientation='180'
# design.components['jj_taper'].options.JJ_up_linewidth='0.20um'
# design.components['jj_taper'].options.JJ_lower_linewidth='0.20um'
# gui.rebuild()
# a_gds.export_to_gds('gds/temp/JJ_taper_'+str(round(0.20,2))+'r_test.gds')

09:53AM 30s WARNING [_qgeometry_to_gds]: Unexpected shapely object geometry.The variable qgeometry_element is <class 'numpy.float64'>, method can currently handle Polygon and FlexPath.
09:53AM 30s WARNING [_import_junction_gds_file]: Not able to find file:"".  Not used to replace junction. Checked directory:"D:\Users\ls\PycharmProjects\pythonProject".
09:53AM 30s WARNING [_qgeometry_to_gds]: Unexpected shapely object geometry.The variable qgeometry_element is <class 'numpy.float64'>, method can currently handle Polygon and FlexPath.
09:53AM 30s WARNING [_import_junction_gds_file]: Not able to find file:"".  Not used to replace junction. Checked directory:"D:\Users\ls\PycharmProjects\pythonProject".
09:53AM 30s WARNING [_qgeometry_to_gds]: Unexpected shapely object geometry.The variable qgeometry_element is <class 'numpy.float64'>, method can currently handle Polygon and FlexPath.
09:53AM 30s WARNING [_import_junction_gds_file]: Not able to find file:"".  Not used to replace junction. C

1

In [4]:
design.delete_all_components()
# 生成9个不同的结线宽gds版图
JJ_up_linewidth = JJ_lower_linewidth= str(0.2)+'um'
jj_path = ''
a_gds = writer_planar(design,jj_path)
jj = JJTaper(design, 'jj_taper', options=Dict(JJ_space_x='0.15um',JJ_space_y='0.8um',fillet='0um',
                                               JJ_up_linewidth=JJ_up_linewidth,JJ_lower_linewidth=JJ_lower_linewidth,
                                               orientation_angle='0',chip='main'))
for i in range(9):
    design.components['jj_taper'].options.JJ_up_linewidth=str(0.2+i*0.01)+'um'
    design.components['jj_taper'].options.JJ_lower_linewidth=str(0.2+i*0.01)+'um'
    gui.rebuild()
    a_gds.export_to_gds('gds/temp/JJ_taper_'+str(round(0.2+i*0.01,2))+'_test.gds')
    design.components['jj_taper'].options.orientation='180'
    gui.rebuild()
    a_gds.export_to_gds('gds/temp/JJ_taper_'+str(round(0.2+i*0.01,2))+'r_test.gds')

# #其中4号和5号比特由于是反向的设置为反向结，选择180度
# for i in range(9):
#     design.components['jj_taper'].options.orientation='180'
#     design.components['jj_taper'].options.JJ_up_linewidth=str(0.2+i*0.01)+'um'
#     design.components['jj_taper'].options.JJ_lower_linewidth=str(0.2+i*0.01)+'um'
#     gui.rebuild()
#     a_gds.export_to_gds('gds/temp/JJ_taper_'+str(round(0.2+i*0.01,2))+'r_test.gds')

04:42PM 06s WARNING [_qgeometry_to_gds]: Unexpected shapely object geometry.The variable qgeometry_element is <class 'numpy.float64'>, method can currently handle Polygon and FlexPath.
04:42PM 06s WARNING [_import_junction_gds_file]: Not able to find file:"".  Not used to replace junction. Checked directory:"D:\Users\ls\PycharmProjects\quantum chip".
04:42PM 06s WARNING [_qgeometry_to_gds]: Unexpected shapely object geometry.The variable qgeometry_element is <class 'numpy.float64'>, method can currently handle Polygon and FlexPath.
04:42PM 06s WARNING [_import_junction_gds_file]: Not able to find file:"".  Not used to replace junction. Checked directory:"D:\Users\ls\PycharmProjects\quantum chip".
04:42PM 07s WARNING [_qgeometry_to_gds]: Unexpected shapely object geometry.The variable qgeometry_element is <class 'numpy.float64'>, method can currently handle Polygon and FlexPath.
04:42PM 07s WARNING [_import_junction_gds_file]: Not able to find file:"".  Not used to replace junction. Che

### 1.2 结文件
将上述gds分文件以cell的形式整合为结文件以供单比特版图使用,这里整合为两个结文件，如下：
1. 结文件中只有漏洞结，但是有N种不同的结宽（这里有5个比特，故N=5）
2. 结文件中有两种不同类型的结，分别是漏斗结和多兰结（或者十字结），均有正反两种类型
### 注：在生成结文件后需要将其放入resources文件夹中，否则无法生效

In [6]:
# #第一个结文件，五个漏洞结不同的线宽
# for i in range(1,6):
#     if i==4 or i==5:
#             globals()[f"gds{i}"]=gdspy.GdsLibrary(infile='gds/temp/JJ_taper_'+str(round(0.15+i*0.01,2))+'r_test.gds', unit=1e-03, precision=1e-13)
#     else:
#             globals()[f"gds{i}"]=gdspy.GdsLibrary(infile='gds/temp/JJ_taper_'+str(round(0.15+i*0.01,2))+'_test.gds', unit=1e-03, precision=1e-13)
#     globals()[f"top_main_cell{i}"]=globals()[f"gds{i}"].cells['TOP_main_1'] 
#     globals()[f"gds{i}"].rename_cell(globals()[f"top_main_cell{i}"], 'FakeJunction_0'+str(i))    
# 
# # Create a new GDS library to hold the combined cells
# combined_gds = gdspy.GdsLibrary(unit=1e-03, precision=1e-13)
# 
# for i in range(1,6):
#     combined_gds.add(globals()[f"top_main_cell{i}"])
# 
# # Save the combined library to a new GDS file
# # combined_gds.write_gds('resources/Fake_Junctions.GDS')
# combined_gds.write_gds('gds/temp/Fake_Junctions1.GDS')
# print("GDS files combined successfully into 'Fake_Junctions1.GDS'")
# 
# #第二个结文件，两种不同类型的结
# gds1 = gdspy.GdsLibrary(infile='gds/temp/JJ_taper_0.18_test.gds', unit=1e-03, precision=1e-13)
# top_main_cell1 = gds1.cells['TOP_main_1']
# gds1.rename_cell(top_main_cell1, 'FakeJunction_01')
# 
# gds2 = gdspy.GdsLibrary(infile='./JJ_taper_0.18r_test.gds', unit=1e-03, precision=1e-13)
# top_main_cell2 = gds2.cells['TOP_main_1']
# gds2.rename_cell(top_main_cell2, 'FakeJunction_02')
# 
# gds3 = gdspy.GdsLibrary(infile='./JJ_Dolan.gds',unit=1e-03, precision=1e-13)
# top_main_cell3 = gds3.cells['TOP_main_1']
# gds3.rename_cell(top_main_cell3, 'FakeJunction_03')
# 
# gds4 = gdspy.GdsLibrary(infile='./JJ_Dolan_flip.gds',unit=1e-03, precision=1e-13)
# top_main_cell4 = gds4.cells['TOP_main_1']
# gds4.rename_cell(top_main_cell4, 'FakeJunction_04')
# 
# # Create a new GDS library to hold the combined cells
# combined_gds = gdspy.GdsLibrary(unit=1e-03, precision=1e-13)
# for i in range(1,5):
#     combined_gds.add(globals()[f"top_main_cell{i}"])
#     
# combined_gds.write_gds('Fake_Junctions.GDS')
# print("GDS files combined successfully into 'Fake_Junctions.GDS'")

GDS files combined successfully into 'Fake_Junctions1.GDS'
GDS files combined successfully into 'Fake_Junctions.GDS'


In [5]:
combined_gds = gdspy.GdsLibrary(unit=1e-03, precision=1e-13)

for i in range(1,10):
    globals()[f"gds{i}"]=gdspy.GdsLibrary(infile='gds/temp/JJ_taper_'+str(round(0.19+i*0.01,2))+'_test.gds', unit=1e-03, precision=1e-13)
    globals()[f"top_main_cell{i}"]=globals()[f"gds{i}"].cells['TOP_main_1'] 
    globals()[f"gds{i}"].rename_cell(globals()[f"top_main_cell{i}"], 'FakeJunction_0'+str(i))    
for i in range(1,10):
    combined_gds.add(globals()[f"top_main_cell{i}"])
for i in range(1,10):
    globals()[f"gds{i}"]=gdspy.GdsLibrary(infile='gds/temp/JJ_taper_'+str(round(0.19+i*0.01,2))+'r_test.gds', unit=1e-03, precision=1e-13)
    globals()[f"top_main_cell{i}"]=globals()[f"gds{i}"].cells['TOP_main_1'] 
    globals()[f"gds{i}"].rename_cell(globals()[f"top_main_cell{i}"], 'FakeJunction_0_r'+str(i))    
for i in range(1,10):
    combined_gds.add(globals()[f"top_main_cell{i}"])

combined_gds.write_gds('Fake_Junctions.GDS')
print("GDS files combined successfully into 'Fake_Junctions.GDS'")

GDS files combined successfully into 'Fake_Junctions.GDS'


## 2.分版图设计
在完成约瑟夫森结文件后，需要设计各个单比特版图，即几何参数不同的版图以测试不同参数对指标（如T1，T2和读取保真度）的影响。
### 2.1 初始化版图设计
设计单比特分版图并对其几何参数进行初始化，其中jj_dict是每个比特对应的约瑟夫森结名字字典。
分版图按约瑟夫森结分为3个类型
1. 5个比特是不同频率，因此采用相同结（均是漏洞结）而不比较不同结的影响，着重比较频率对指标的影响；
2. 5个比特是相同频率，因此采用不同类型的结，比较不同类型结对指标的影响；
3. 5个比特是相同频率，采用不同线宽的结，比较不同线宽对频率的影响。

In [4]:
# 版图1-3 不同频率，1-5 qubits 相同结
jj_dict = {'Q1':'FakeJunction_03','Q2':'FakeJunction_03','Q3':'FakeJunction_03','Q4':'FakeJunction_03_r',
           'Q5':'FakeJunction_03_r','Q6':'FakeJunction_01','Q7':'FakeJunction_02','Q8':'FakeJunction_05',
           'Q9':'my_other_junction'} 
# jj_dict = {'Q1':'FakeJunction_01','Q2':'FakeJunction_01','Q3':'FakeJunction_01','Q4':'FakeJunction_02',
#            'Q5':'FakeJunction_02','Q6':'FakeJunction_01','Q7':'FakeJunction_01','Q8':'FakeJunction_01',
#            'Q9':'FakeJunction_01'}  
# 
# # 版图4-8 相同频率，1-5 qubit 不同结
# # jj_dict0 = {'Q1':'FakeJunction_03','Q2':'FakeJunction_04','Q3':'FakeJunction_03','Q4':'FakeJunction_04_r',
# #            'Q5':'FakeJunction_03_r','Q6':'FakeJunction_01','Q7':'FakeJunction_02','Q8':'FakeJunction_05',
# #            'Q9':'my_other_junction'}
# jj_dict0 = {'Q1':'FakeJunction_01','Q2':'FakeJunction_03','Q3':'FakeJunction_01','Q4':'FakeJunction_04',
#            'Q5':'FakeJunction_02','Q6':'FakeJunction_01','Q7':'FakeJunction_01','Q8':'FakeJunction_01',
#            'Q9':'FakeJunction_01'} 
# 
# #版图9 相同频率， 1-5 qubit 线宽不同结
# jj_dict1 = {'Q1':'FakeJunction_01','Q2':'FakeJunction_02','Q3':'FakeJunction_03','Q4':'FakeJunction_04',
#            'Q5':'FakeJunction_05','Q6':'FakeJunction_03','Q7':'FakeJunction_03','Q8':'FakeJunction_03',
#            'Q9':'FakeJunction_03'}
# 
design,gui= single_qubit_layout(jj_dict)
# design0,gui0 = single_qubit_layout(jj_dict0)
# design1,gui1 = single_qubit_layout(jj_dict1)



  return original_import(name, *args, **kwargs)



In [3]:
for i in range(1,10):
    globals()[f"jj_dict{i}"] = {'Q1':'FakeJunction_0'+str(i),'Q2':'FakeJunction_0'+str(i),'Q3':'FakeJunction_0'+str(i),'Q4':'FakeJunction_0_r'+str(i),
           'Q5':'FakeJunction_0_r'+str(i),'Q6':'FakeJunction_0'+str(i),'Q7':'FakeJunction_0'+str(i),'Q8':'FakeJunction_0'+str(i),
           'Q9':'FakeJunction_0'+str(i)}
    globals()[f"design{i}"],globals()[f"gui{i}"]=single_qubit_layout(globals()[f"jj_dict{i}"])

  return original_import(name, *args, **kwargs)



In [6]:
# #设置参数文件索引为name，设置结位置
# set_index='name'
# jj_path = './resources/Fake_Junctions.GDS'
# jj_path1 = './resources/Fake_Junctions1.GDS'
# #设置输出文件选项
# a_gds = writer_planar(design,jj_path,negative_mask=[1,2])
# a_gds0 = writer_planar(design0,jj_path,negative_mask=[1,2])
# a_gds1 = writer_planar(design1,jj_path1,negative_mask=[1,2])


In [4]:
#设置参数文件索引为name，设置结位置
set_index='name'
jj_path = './resources/Fake_Junctions11.GDS'
#设置输出文件选项
for i in range(1,10):
    globals()[f"a_gds{i}"]=writer_planar(globals()[f"design{i}"],jj_path,negative_mask=[1,2])

### 2.2 读出参数文件存储为JSON文件
在本设计中，参数以excel文件的形式提供并且格式需要是标准化的以供外部读取，具体如下
1. 每一行代表版图中需要改变参数的器件名字，如Q1-Q5， meanderQ1-meanderQ9等，其取名参考初始版图代码中的命名；
2. 每一列代表器件中的几何参数，这里将所有需要改变的器件参数都需要列出来，如果几何参数不属于某个器件，则留白，系统将自动转换为None
   例如 total_length 是谐振腔参数，则在Q1的total_length那一列则留白不填；
3. 几何参数遵循{}嵌套规则，具体如connection_pads{readout{pad_gap}}代表的是比特上和谐振腔耦合处的pad_gap参数，即字典connection_pads:{readout:{pad_gap}}

读取函数将以一定规则读取这个参数文件并将其保存为JSON格式以供后续的参数修改

In [8]:
# for i in range(1,10):
#     globals()[f"path{i}"]='data\coupling parameter'+str(i)+'.xlsx'
#     globals()[f"savedpath{i}"]='data\design'+str(i)+'.json'
#     globals()[f"parameter{i}"]=extract_nested_dict(globals()[f"path{i}"],set_index)
#     save_json_parameter(globals()[f"parameter{i}"],globals()[f"savedpath{i}"])


### 2.3 参数输入
参数输入函数将JSON格式的参数文件导入到初始版图中对分版图参数进行逐一修改，这里分别对三种类型的版图进行修改，同时加入空气桥，生成N个不同参数设置的分版图（这里N=9）


In [ ]:
# list0=[]
# list1=[design.components['cpw_openLeft'], design.components['cpw_openRight']]
# for i in range(1, 10):
#     list0.append(design.components['meanderQ'+str(i)])
# for i in range(1, 9):
#     list1.append(design.components['cpw_'+str(i)])    
#     
# for i in range(1,4):
#     parameter_import(design,load_json(globals()[f"savedpath{i}"]))
#     gui.rebuild()
#     build_airbridge(design,gui,list_meander=list0+list1)  
#     # a_gds.options.gds_unit=1
#     a_gds.export_to_gds('design'+str(i)+'.gds')
#     print(f'gds{i} is finished')
# for i in range(4,9):
#     parameter_import(design0,load_json(globals()[f"savedpath{i}"]))
#     gui0.rebuild()
#     build_airbridge(design0,gui0,list_meander=list0+list1)
#     # a_gds0.options.gds_unit=1
#     a_gds0.export_to_gds('design'+str(i)+'.gds')
#     print(f'gds{i} is finished')
#     
# parameter_import(design1,load_json(globals()[f"savedpath{9}"]))
# gui1.rebuild()
# build_airbridge(design1,gui1,list_meander=list0+list1)    
# a_gds1.export_to_gds('design'+str(9)+'.gds')
# print(f'gds{9} is finished')

In [6]:
for i in range(1,7):
    # parameter_import(globals()[f"design{i}"],convert_keys_to_int(load_json('data\design'+str(7)+'.json')))
    parameter_import(globals()[f"design{i}"],convert_keys_to_int(load_json('data\single_qubit_option'+str(9)+'.json')))
    globals()[f"gui{i}"].rebuild()
    # build_airbridge(globals()[f"design{i}"],globals()[f"gui{i}"],list_meander=list0+list1)  
    # a_gds.options.gds_unit=1
    globals()[f"a_gds{i}"].export_to_gds('gds/temp/design'+str(i)+'.gds')
    print(f'gds{i} is finished')
for i in range(7,10):
    # parameter_import(globals()[f"design{i}"],convert_keys_to_int(load_json('data\design'+str(7)+'.json')))
    parameter_import(globals()[f"design{i}"],convert_keys_to_int(load_json('data\single_qubit_option'+str(9.1)+'.json')))
    globals()[f"gui{i}"].rebuild()
    # build_airbridge(globals()[f"design{i}"],globals()[f"gui{i}"],list_meander=list0+list1)  
    # a_gds.options.gds_unit=1
    globals()[f"a_gds{i}"].export_to_gds('gds/temp/design'+str(i)+'.gds')
    print(f'gds{i} is finished')

11:35AM 23s WARNING [_import_junction_gds_file]: Not able to find file:"./resources/Fake_Junctions11.GDS".  Not used to replace junction. Checked directory:"D:\Users\ls\PycharmProjects\quantum chip\transmon qubit coherence project\resources".


gds1 is finished


11:35AM 26s WARNING [_import_junction_gds_file]: Not able to find file:"./resources/Fake_Junctions11.GDS".  Not used to replace junction. Checked directory:"D:\Users\ls\PycharmProjects\quantum chip\transmon qubit coherence project\resources".


gds2 is finished


11:35AM 29s WARNING [_import_junction_gds_file]: Not able to find file:"./resources/Fake_Junctions11.GDS".  Not used to replace junction. Checked directory:"D:\Users\ls\PycharmProjects\quantum chip\transmon qubit coherence project\resources".


...

gds7 is finished


11:35AM 42s WARNING [_import_junction_gds_file]: Not able to find file:"./resources/Fake_Junctions11.GDS".  Not used to replace junction. Checked directory:"D:\Users\ls\PycharmProjects\quantum chip\transmon qubit coherence project\resources".


gds8 is finished


11:35AM 44s WARNING [_import_junction_gds_file]: Not able to find file:"./resources/Fake_Junctions11.GDS".  Not used to replace junction. Checked directory:"D:\Users\ls\PycharmProjects\quantum chip\transmon qubit coherence project\resources".


gds9 is finished


将分版图统一单位，即将user_units dbuu设置为0.001

In [7]:
import klayout.db as db
for i in range(1,10):
    layout = db.Layout()
    layout.read('gds/temp/design'+str(i)+'.gds')
    db.SaveLayoutOptions.gds2_user_units=0.001
    layout.write('gds/temp/design'+str(i)+'.gds')
# layout = db.Layout()
# layout.read('design'+str(9)+'.gds')
# db.SaveLayoutOptions.gds2_user_units=0.001
# layout.write('design'+str(9)+'.gds')

### 2.4 磁通陷阱导入
创建磁通陷阱gds文件，将其加入到各自的分版图中（其中分版图8由于要比较有无磁通陷阱的影响则不加磁通陷阱）


In [8]:
# Load the GDS file
template = gdspy.GdsLibrary(infile='gds/temp/template.gds')
gds1 = gdspy.GdsLibrary(infile='gds/temp/design1.gds')
gds2 = gdspy.GdsLibrary(infile='gds/temp/design2.gds')
gds3 = gdspy.GdsLibrary(infile='gds/temp/design3.gds')
gds4 = gdspy.GdsLibrary(infile='gds/temp/design4.gds')
gds5 = gdspy.GdsLibrary(infile='gds/temp/design5.gds')
gds6 = gdspy.GdsLibrary(infile='gds/temp/design6.gds')
gds7 = gdspy.GdsLibrary(infile='gds/temp/design7.gds')
gds8 = gdspy.GdsLibrary(infile='gds/temp/design8.gds')
gds9 = gdspy.GdsLibrary(infile='gds/temp/design9.gds')
# label = gdspy.GdsLibrary(infile='label.gds')
# cheese = gdspy.GdsLibrary(infile='cheese.gds')

In [10]:
# # cell_add(cheese,gds1,gds_cell_list=['TOP1'])
# # gds1.write_gds('design1.gds')
# cell_cheese = cheese.cells['TOP1']
# for layer,polygon in cell_cheese.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1])
#     gds1.cells['TOP'].add(polygon_set)
# gds1.write_gds('design1.gds')
# 
# for layer,polygon in cell_cheese.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1])
#     gds2.cells['TOP'].add(polygon_set)
# gds2.write_gds('design2.gds')
# 
# for layer,polygon in cell_cheese.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1])
#     gds3.cells['TOP'].add(polygon_set)
# gds3.write_gds('design3.gds')
# # 
# cell_cheese = cheese.cells['TOP']
# for layer,polygon in cell_cheese.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1])
#     gds4.cells['TOP'].add(polygon_set)
# gds4.write_gds('design4.gds')
# 
# for layer,polygon in cell_cheese.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1])
#     gds5.cells['TOP'].add(polygon_set)
# gds5.write_gds('design5.gds')
# 
# for layer,polygon in cell_cheese.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1])
#     gds6.cells['TOP'].add(polygon_set)
# gds6.write_gds('design6.gds')
# 
# for layer,polygon in cell_cheese.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1])
#     gds7.cells['TOP'].add(polygon_set)
# gds7.write_gds('design7.gds')
# 
# for layer,polygon in cell_cheese.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1])
#     gds9.cells['TOP'].add(polygon_set)
# gds9.write_gds('design9.gds')


## 3.总版图生成
### 3.1 加入标志
这里加入标志的方法有两种，第一种是先创建分版图标志然后通过程序控制在总版图的位置；第二种是直接创建总版图标志模版，然后将分版图放入总版图模版中
本设计先采用第二种方法

In [9]:
delta_x = 9600
delta_y = 8000
    
cell_add(gds1,template,-delta_x,delta_y)
cell_add(gds2,template,0,delta_y)
cell_add(gds3,template,delta_x,delta_y)
cell_add(gds4,template,-delta_x,0)
cell_add(gds5,template,0,0)
cell_add(gds6,template,delta_x,0)
cell_add(gds7,template,-delta_x,-delta_y)
cell_add(gds8,template,0,-delta_y)
cell_add(gds9,template,delta_x,-delta_y)




# cell1 = gds1.cells['TOP_main_1']
# for layer,polygon in cell1.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(-delta_x,delta_y)        
#     template.cells['TOP'].add(polygon_set)
# cell1 = gds1.cells['TOP_main_2']
# for layer,polygon in cell1.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(-delta_x,delta_y)        
#     template.cells['TOP'].add(polygon_set)
# cell1 = gds1.cells['TOP_main_3']
# for layer,polygon in cell1.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(-delta_x,delta_y)        
#     template.cells['TOP'].add(polygon_set)

# cell2 = gds2.cells['TOP']
# for layer,polygon in cell2.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(0,delta_y)        
#     template.cells['TOP'].add(polygon_set)
# 
# cell3 = gds3.cells['TOP']
# for layer,polygon in cell3.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(delta_x,delta_y)        
#     template.cells['TOP'].add(polygon_set)
# 
# cell4 = gds4.cells['TOP']
# for layer,polygon in cell4.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(-delta_x,0)        
#     template.cells['TOP'].add(polygon_set)
# 
# cell5 = gds5.cells['TOP']
# for layer,polygon in cell5.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1])       
#     template.cells['TOP'].add(polygon_set)
# 
# cell6 = gds6.cells['TOP']
# for layer,polygon in cell6.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(delta_x,0)        
#     template.cells['TOP'].add(polygon_set)
# 
# cell7 = gds7.cells['TOP']
# for layer,polygon in cell7.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(-delta_x,-delta_y)        
#     template.cells['TOP'].add(polygon_set)
# 
# cell8 = gds8.cells['TOP']
# for layer,polygon in cell8.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(0,-delta_y)        
#     template.cells['TOP'].add(polygon_set)
# 
# cell9 = gds9.cells['TOP']
# for layer,polygon in cell9.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(delta_x,-delta_y)        
#     template.cells['TOP'].add(polygon_set)

# label_x = -4000
# label_y = 2500
# label_cell = label.cells['TOP']
# for layer,polygon in label_cell.get_polygons(by_spec=True).items():
#     polygon_set1 = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(label_x-delta_x,label_y+delta_y)
#     polygon_set2 = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(label_x,label_y+delta_y)    
#     polygon_set3 = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(label_x+delta_x,label_y+delta_y)    
#     polygon_set4 = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(label_x-delta_x,label_y)    
#     polygon_set5 = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(label_x,label_y)    
#     polygon_set6 = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(label_x+delta_x,label_y)    
#     polygon_set7 = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(label_x-delta_x,label_y-delta_y)    
#     polygon_set8 = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(label_x,label_y-delta_y)    
#     polygon_set9 = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(label_x+delta_x,label_y-delta_y)        
#     gds5.top_level()[0].add([polygon_set1,polygon_set2,polygon_set3,polygon_set4,polygon_set5,polygon_set6,polygon_set7,polygon_set8,polygon_set9])  
template.write_gds('design_combined.gds')
print("GDS files combined successfully generated")


KeyError: 'TOP_main_2'

In [10]:
delta_x = 9600
delta_y = 8000
gds_cell_list=['TOP_main_1', ]    
cell_add(gds1,template,-delta_x,delta_y,gds_cell_list=gds_cell_list)
cell_add(gds2,template,0,delta_y,gds_cell_list=gds_cell_list)
cell_add(gds3,template,delta_x,delta_y,gds_cell_list=gds_cell_list)
cell_add(gds4,template,-delta_x,0,gds_cell_list=gds_cell_list)
cell_add(gds5,template,0,0,gds_cell_list=gds_cell_list)
cell_add(gds6,template,delta_x,0,gds_cell_list=gds_cell_list)
cell_add(gds7,template,-delta_x,-delta_y,gds_cell_list=gds_cell_list)
cell_add(gds8,template,0,-delta_y,gds_cell_list=gds_cell_list)
cell_add(gds9,template,delta_x,-delta_y,gds_cell_list=gds_cell_list)
template.write_gds('design_combined.gds')
print("GDS files combined successfully generated")

GDS files combined successfully generated
